
# Matryoshka embeddings on Colab — MRL / ESE / MIPIC / SDR-MRL

Clone [`duncan-nguyen/embedding-mrl`](https://github.com/duncan-nguyen/embedding-mrl),
pick a method and a backbone in the form below, and run one experiment end to end.

**How to use**
1. `Runtime → Change runtime type → GPU` (T4 is enough for BERT / TinyBERT).
2. Set everything in **§1 Experiment settings** — nothing below needs editing.
3. `Runtime → Run all`.

Training data and every evaluation CSV ship inside the repo, so the only
download at runtime is the backbone from the Hugging Face Hub.

> The SDR-MRL method (`configs/sdr/*`) has to exist on the branch you clone.
> If `git clone` succeeds but §4 reports a missing config, push that work first.

## 1. Experiment settings

In [2]:

#@title  Experiment settings { display-mode: "form" }

#@markdown ### Method and backbone
METHOD = "mipic"  #@param ["mrl", "ese", "mipic", "sdr"]
BACKBONE = "bert"  #@param ["bert", "tinybert_6l", "bgem3", "qwen3_0.6b"]
#@markdown `bert` (110M) and `tinybert_6l` (67M) fit a free T4 comfortably.
#@markdown `bgem3` (568M) and `qwen3_0.6b` (596M) need a smaller batch, and
#@markdown Qwen3 also needs `trust_remote_code`, which the config already sets.

#@markdown ### Training  &nbsp; *(Table 7 defaults: 5 epochs, lr 2e-5, batch 64, len 256)*
EPOCHS = 5  #@param {type:"integer"}
BATCH_SIZE = 64  #@param {type:"integer"}
LEARNING_RATE = 2e-5  #@param {type:"number"}
MAX_LENGTH = 256  #@param {type:"integer"}
SEED = 42  #@param {type:"integer"}
FP16 = True  #@param {type:"boolean"}
MAX_GRAD_NORM = 1.0  #@param {type:"number"}
#@markdown `MAX_TRAIN_SAMPLES = 0` uses the full 20 244-sentence corpus.
#@markdown Set it to e.g. 2000 for a quick smoke run first.
MAX_TRAIN_SAMPLES = 0  #@param {type:"integer"}

#@markdown ### Evaluation
EVAL_SPLIT = "test"  #@param ["test", "validation"]
#@markdown The full suite is slow; leave `EVAL_EVERY_EPOCH` off to score once at the end.
EVAL_EVERY_EPOCH = False  #@param {type:"boolean"}
#@markdown Semantic distortion-rate protocol (SDR-MRL §6). Runs for *every* method,
#@markdown which is the point — it compares where each one puts semantic information.
SEMANTIC_DISTORTION = True  #@param {type:"boolean"}
ROTATION_TRIALS = 2  #@param {type:"integer"}
#@markdown Optional fixed reference teacher (Eq 86). Leave blank to score each model
#@markdown against its own full width, which measures self-consistency instead.
REFERENCE_MODEL = ""  #@param {type:"string"}

#@markdown ### SDR-MRL knobs &nbsp; *(ignored unless `METHOD = "sdr"`)*
LAMBDA_SEM = 1.0  #@param {type:"number"}
#@markdown `LAMBDA_MONO = 0` is Eq 56's minimal model. Turn it on only if the
#@markdown diagnostics in §8 report `V_mono > 0`; Eq 107 sweeps {0.05, 0.1, 0.3}.
LAMBDA_MONO = 0.1  #@param {type:"number"}
TAU_TEACHER = 0.05  #@param {type:"number"}
TAU_STUDENT = 0.05  #@param {type:"number"}
SDR_TEACHER = "online"  #@param ["online", "ema", "frozen"]
SDR_TEACHER_MODEL = ""  #@param {type:"string"}
SDR_GEOMETRY = "snd"  #@param ["snd", "gram_mse", "cka", "hard_neighbor"]
SDR_DIVERGENCE = "forward_kl"  #@param ["forward_kl", "reverse_kl", "js"]
SDR_CANDIDATES = "all"  #@param ["all", "teacher_topm", "teacher_topm_student_hard"]
SDR_RATE_PRIOR = "uniform"  #@param ["uniform", "inverse_dim"]
#@markdown Sample one rate per step instead of all prefixes — unbiased (Eq 68), cheaper.
SDR_STOCHASTIC_RATE = False  #@param {type:"boolean"}
#@markdown Log the mathematical diagnostics every N steps (0 = off).
DIAGNOSTICS_EVERY = 50  #@param {type:"integer"}

#@markdown ### Repository and output
REPO_URL = "https://github.com/duncan-nguyen/embedding-mrl"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
RUN_NAME = ""  #@param {type:"string"}
#@markdown Copy the finished run to `MyDrive/embedding-mrl/` so it survives the session.
SAVE_TO_DRIVE = False  #@param {type:"boolean"}
#@markdown Also run `scripts/verify_sdr_math.py` (§9): 35 checks of the code against the paper's proofs.
VERIFY_MATH = True  #@param {type:"boolean"}

RUN_NAME = RUN_NAME or f"{METHOD}_{BACKBONE}"
print(f"{METHOD.upper()} on {BACKBONE} -> run '{RUN_NAME}'")
print(f"{EPOCHS} epochs, batch {BATCH_SIZE}, lr {LEARNING_RATE}, max_length {MAX_LENGTH}")
if METHOD == "sdr":
    print(f"L = L_task + {LAMBDA_SEM}*sum_k pi_k D_k" +
          (f" + {LAMBDA_MONO}*sum_k [D_k - D_(k-1)]_+" if LAMBDA_MONO > 0 else ""))
    print(f"teacher={SDR_TEACHER}, geometry={SDR_GEOMETRY}, "
          f"tau_T={TAU_TEACHER}, tau_S={TAU_STUDENT}, pi={SDR_RATE_PRIOR}")

MIPIC on bert -> run 'mipic_bert'
5 epochs, batch 64, lr 2e-05, max_length 256


## 2. Runtime check

In [3]:

import subprocess, sys

print("Python", sys.version.split()[0])
try:
    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    print("GPU   ", gpu)
except (FileNotFoundError, subprocess.CalledProcessError):
    gpu = ""
    print("GPU    none — this will run on CPU and take hours.")
    print("       Runtime -> Change runtime type -> T4 GPU, then re-run.")

# The two large backbones do not fit a 16GB card at batch 16 / length 256.
if BACKBONE in ("bgem3", "qwen3_0.6b") and "16" in gpu.split(",")[-1]:
    print(f"\nNote: {BACKBONE} on a 16GB card usually needs BATCH_SIZE 4-8 "
          f"(currently {BATCH_SIZE}) or MAX_LENGTH 128.")

Python 3.13.15
GPU    NVIDIA A100-SXM4-80GB, 81920 MiB


## 3. Clone the repository and install dependencies

In [4]:


import os, subprocess, sys

from pathlib import Path



REPO_DIR = Path("/content/embedding-mrl") if Path("/content").exists() else Path.cwd() / "embedding-mrl"



def run(cmd, **kwargs):

    """Run a command, streaming its output so long steps are not silent."""

    process = subprocess.Popen(

        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,

        text=True, bufsize=1, **kwargs,

    )

    for line in process.stdout:

        print(line, end="")

    process.wait()

    return process.returncode



if not REPO_DIR.exists():

    run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)])

else:

    print(f"{REPO_DIR} already present — pulling instead")

    run(["git", "pull", "--ff-only"], cwd=str(REPO_DIR))



os.chdir(REPO_DIR)

print("\nworking directory:", Path.cwd())

print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)



# Colab already ships torch, pandas, sklearn and scipy; this fills the gaps.

run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "matplotlib"])

Cloning into '/content/embedding-mrl'...

working directory: /content/embedding-mrl
af75704 Update documentation references to the new LaTeX format and add Colab notebook for experiments



0

## 4. Build the run configuration

Everything from §1 becomes a `--set key=value` override on top of the shipped
YAML, so the file on disk is never edited and the resolved config is printed
before anything trains.

In [5]:

from pathlib import Path

CONFIG_PATH = Path("configs") / METHOD / f"{BACKBONE}.yaml"
OUTPUT_DIR = Path("outputs") / RUN_NAME

if not CONFIG_PATH.exists():
    available = sorted(str(p) for p in Path("configs").rglob("*.yaml") if p.name != "base.yaml")
    raise FileNotFoundError(
        f"{CONFIG_PATH} is not on branch '{BRANCH}'.\nAvailable:\n  " + "\n  ".join(available)
    )

def yaml_value(value):
    """Render a Python value as a literal PyYAML reads back as the same type.

    PyYAML is YAML 1.1, whose float resolver needs *both* a decimal point and a
    signed exponent - so a bare `2e-05` comes back as the string "2e-05" and
    lands in `train.lr`. Emit `2.0e-05` instead.
    """
    if isinstance(value, bool):
        return "true" if value else "false"
    if value is None or value == "":
        return "null"
    if isinstance(value, float):
        text = repr(value)
        if "e" in text:
            mantissa, _, exponent = text.partition("e")
            if "." not in mantissa:
                mantissa += ".0"
            if not exponent.startswith(("+", "-")):
                exponent = "+" + exponent
            text = f"{mantissa}e{exponent}"
        return text
    return str(value)

overrides = [
    f"name={RUN_NAME}",
    f"train.output_dir={OUTPUT_DIR}",
    f"train.epochs={EPOCHS}",
    f"train.batch_size={BATCH_SIZE}",
    f"train.lr={yaml_value(float(LEARNING_RATE))}",
    f"train.seed={SEED}",
    f"train.fp16={yaml_value(FP16)}",
    f"train.max_grad_norm={yaml_value(MAX_GRAD_NORM if MAX_GRAD_NORM > 0 else None)}",
    f"data.max_length={MAX_LENGTH}",
    f"eval.split={EVAL_SPLIT}",
    f"eval.every_epoch={yaml_value(EVAL_EVERY_EPOCH)}",
    f"eval.semantic_distortion={yaml_value(SEMANTIC_DISTORTION)}",
    f"eval.rotation_trials={ROTATION_TRIALS}",
]
if MAX_TRAIN_SAMPLES > 0:
    overrides.append(f"data.max_train_samples={MAX_TRAIN_SAMPLES}")
if REFERENCE_MODEL:
    overrides.append(f"eval.reference_model={REFERENCE_MODEL}")

if METHOD == "sdr":
    overrides += [
        f"sdr.lambda_sem={yaml_value(float(LAMBDA_SEM))}",
        f"sdr.lambda_mono={yaml_value(float(LAMBDA_MONO))}",
        f"sdr.teacher_temperature={yaml_value(float(TAU_TEACHER))}",
        f"sdr.student_temperature={yaml_value(float(TAU_STUDENT))}",
        f"sdr.teacher={SDR_TEACHER}",
        f"sdr.geometry={SDR_GEOMETRY}",
        f"sdr.divergence={SDR_DIVERGENCE}",
        f"sdr.candidates={SDR_CANDIDATES}",
        f"sdr.rate_prior={SDR_RATE_PRIOR}",
        f"sdr.stochastic_rate={yaml_value(SDR_STOCHASTIC_RATE)}",
        f"sdr.diagnostics_every={DIAGNOSTICS_EVERY}",
    ]
    if SDR_TEACHER == "frozen":
        if not SDR_TEACHER_MODEL:
            raise ValueError("SDR_TEACHER='frozen' needs SDR_TEACHER_MODEL (a checkpoint id or path)")
        overrides.append(f"sdr.teacher_model={SDR_TEACHER_MODEL}")

TRAIN_ARGS = [str(a) for pair in (("--set", o) for o in overrides) for a in pair]
print(f"config: {CONFIG_PATH}\noverrides:")
for override in overrides:
    print("   ", override)

print("\n" + "=" * 60 + "\nresolved configuration\n" + "=" * 60)
run([sys.executable, "scripts/train.py", "--config", str(CONFIG_PATH), *TRAIN_ARGS, "--print-config"])

config: configs/mipic/bert.yaml
overrides:
    name=mipic_bert
    train.output_dir=outputs/mipic_bert
    train.epochs=5
    train.batch_size=64
    train.lr=2e-05
    train.seed=42
    train.fp16=true
    train.max_grad_norm=1.0
    data.max_length=256
    eval.split=test
    eval.every_epoch=false
    eval.semantic_distortion=true
    eval.rotation_trials=2

resolved configuration
name: mipic_bert
method: mipic
model:
  name_or_path: google-bert/bert-base-uncased
  hidden_dim: 768
  pooling: cls
  trust_remote_code: false
  torch_dtype: null
data:
  root: /content/embedding-mrl/data
  train_file: train/final_data.csv
  test_dir: test
  text_column: text
  max_length: 256
  num_workers: 2
  max_train_samples: null
train:
  epochs: 5
  batch_size: 64
  lr: 2e-05
  weight_decay: 0.01
  warmup_ratio: 0.1
  min_lr: 2.0e-06
  scheduler: cosine_with_min_lr
  max_grad_norm: 1.0
  seed: 42
  fp16: true
  output_dir: outputs/mipic_bert
  save_model: true
  empty_cache_every: 0
matryoshka:
  d

0

## 5. Train

The backbone downloads on first use. With BERT-base, the full corpus and the
Table 7 defaults this is roughly 1 265 steps per epoch.

In [6]:

import time

started = time.time()
exit_code = run([sys.executable, "scripts/train.py", "--config", str(CONFIG_PATH), *TRAIN_ARGS])
elapsed = time.time() - started

print(f"\nfinished in {elapsed / 60:.1f} min (exit code {exit_code})")
if exit_code != 0:
    raise RuntimeError("training failed — see the log above")

10:02:13 | INFO    | Experiment mipic_bert (method=mipic)
10:02:27 | INFO    | NumExpr defaulting to 12 threads.
10:02:30 | INFO    | Loading google-bert/bert-base-uncased
10:02:31 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5272.39it/s]
[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias     

RuntimeError: training failed — see the log above

## 6. Results

In [ ]:

import json
import pandas as pd

report = json.loads((OUTPUT_DIR / "results.json").read_text())

print(f"{report['experiment']['name']}  ({report['experiment']['method']}, "
      f"{report['experiment']['model']})")
print(f"trained {report['training']['epochs_completed']} epochs, "
      f"final loss {report['training']['final_loss']:.4f}, "
      f"{report['training']['duration_seconds'] or 0:.0f}s")

table = pd.read_csv(OUTPUT_DIR / "results.csv").set_index("dim")
pd.set_option("display.width", 200, "display.max_columns", 50)
table

## 7. Quality and distortion against the storage rate

The whole point of a Matryoshka model is the shape of these curves, not the
score at any single width.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# A small, validated palette: three categorical hues that stay separable under
# deuteranopia and tritanopia, and a single-hue ordinal ramp for the prefixes.
SURFACE, INK, INK_MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e7e6e2"
CATEGORICAL = ["#2a78d6", "#eb6834", "#1baf7a"]
RATE_RAMP = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#104281"]

def new_axes(title, xlabel, ylabel, subtitle=None, size=(7.6, 4.3)):
    figure, ax = plt.subplots(figsize=size, facecolor=SURFACE)
    ax.set_facecolor(SURFACE)
    ax.set_title(title, color=INK, fontsize=13, fontweight="600", loc="left",
                 pad=20 if subtitle else 10)
    if subtitle:
        ax.text(0, 1.035, subtitle, transform=ax.transAxes, color=INK_MUTED,
                fontsize=9, va="bottom")
    ax.set_xlabel(xlabel, color=INK_MUTED, fontsize=10)
    ax.set_ylabel(ylabel, color=INK_MUTED, fontsize=10)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=INK_MUTED, labelsize=9.5, length=0)
    return figure, ax

def label_line_ends(ax, x, entries):
    """Direct labels at the right edge, nudged apart so converging lines stay readable.

    Labels wear the ink token, never the series colour — the mark beside them
    already carries identity.
    """
    low, high = ax.get_ylim()
    minimum_gap = (high - low) * 0.06
    placed = []
    for y, text in sorted(entries, key=lambda entry: entry[0]):
        if placed and y - placed[-1][0] < minimum_gap:
            y = placed[-1][0] + minimum_gap
        placed.append((y, text))
    for y, text in placed:
        ax.annotate(text, (x, y), xytext=(8, 0), textcoords="offset points",
                    color=INK, fontsize=9.5, va="center", annotation_clip=False)

def rate_axis(ax, dims):
    ax.set_xscale("log", base=2)
    ax.set_xticks(dims)
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{int(v)}"))
    ax.xaxis.set_minor_formatter(ticker.NullFormatter())

DIMS = report["experiment"]["matryoshka_dims"]
LABELS = {"classification": "Classification", "sts": "STS", "pair": "Pair"}

figure, ax = new_axes(
    "Task quality against embedding width",
    "embedding dimension (log scale)",
    "mean score across tasks",
    subtitle=f"{report['experiment']['method'].upper()} · {report['experiment']['model']}",
)
ends = []
for index, (family, per_dim) in enumerate(report["summary"].items()):
    values = [per_dim[f"dim_{d}"] for d in DIMS]
    ax.plot(DIMS, values, color=CATEGORICAL[index % len(CATEGORICAL)], linewidth=2,
            marker="o", markersize=6, markeredgecolor=SURFACE, markeredgewidth=1.4,
            label=LABELS.get(family, family), zorder=3)
    ends.append((values[-1], f"{LABELS.get(family, family)}  {values[-1]:.3f}"))

rate_axis(ax, DIMS)
ax.set_xlim(DIMS[0] * 0.85, DIMS[-1] * 2.9)
# Both a legend and direct labels: identity is never carried by colour alone.
label_line_ends(ax, DIMS[-1], ends)
ax.legend(frameon=False, loc="lower right", fontsize=9.5, labelcolor=INK_MUTED)
plt.tight_layout()
plt.show()

In [ ]:
semantic = report.get("semantic")

if not semantic:
    print("No semantic block — re-run with SEMANTIC_DISTORTION = True.")
else:
    normalized = {int(k.split("_")[1]): v for k, v in semantic["normalized_distortion"].items()}
    full_dim = report["experiment"]["hidden_dim"]
    dims = sorted(normalized)
    rates = [0.0] + [d / full_dim for d in dims]
    values = [1.0] + [normalized[d] for d in dims]

    figure, ax = new_axes(
        "Semantic distortion-rate curve",
        "storage rate  $r_k = d_k / D$",
        "normalised distortion  $\\tilde{\\mathcal{D}}_k$",
        subtitle=(f"reference teacher: {semantic['reference']} · "
                  f"{semantic['corpus_size']} sentences · "
                  f"SDRA = {semantic['sdra']:.4f} (lower is better)"),
    )
    # The shaded area *is* SDRA (Eq 91), so it is worth drawing rather than decorating.
    ax.fill_between(rates, values, color=CATEGORICAL[0], alpha=0.12, zorder=1)
    ax.plot(rates, values, color=CATEGORICAL[0], linewidth=2, marker="o", markersize=6,
            markeredgecolor=SURFACE, markeredgewidth=1.4, zorder=3)
    ax.axhline(1.0, color=INK_MUTED, linewidth=1, linestyle=(0, (4, 3)), zorder=2)

    top = max(values)
    ax.set_ylim(min(0, min(values)) - top * 0.06, top * 1.22)
    ax.set_xlim(-0.04, 1.08)
    ax.annotate("zero-rate decoder — a prefix above this line\nis worse than knowing nothing",
                (1.06, 1.0), xytext=(0, 8), textcoords="offset points",
                color=INK_MUTED, fontsize=9, ha="right")

    # Label the endpoints and anything that loses to the zero-rate decoder,
    # placing each label on whichever side of the curve is free.
    for dim, rate, value in zip(dims, rates[1:], values[1:]):
        if dim not in (dims[0], dims[-1]) and value <= 1.0:
            continue
        # Above when it would otherwise land on the axis or under the curve.
        above = value > 1.0 or value < top * 0.15
        ax.annotate(f"d={dim}", (rate, value),
                    xytext=(0, 11 if above else -15), textcoords="offset points",
                    color=INK, fontsize=9,
                    ha="right" if rate > 0.95 else "center")
    plt.tight_layout()
    plt.show()

    print(f"V_mono (Eq 108) = {semantic['monotonicity_violation_rate']:.3f}"
          "  — the fraction of prefixes where adding coordinates made the "
          "neighbourhood harder to recover.")
    if semantic["monotonicity_violation_rate"] > 0:
        print("  > 0, so sdr.lambda_mono has something to fix: sweep {0.05, 0.1, 0.3}.")
    else:
        print("  = 0, so the monotonic regulariser would be an inactive hinge — "
              "leave lambda_mono at 0.")

In [ ]:

if semantic:
    gains = {int(k.split("_")[1]): v for k, v in semantic["refinement_gain"].items()}

    figure, ax = new_axes(
        "Semantic gain per added coordinate",
        "prefix the block completes",
        r"$\eta_k = (\mathcal{D}_{k-1} - \mathcal{D}_k) / (d_k - d_{k-1})$",
        subtitle="Where this model chooses to spend capacity (Eq 122)",
    )
    positions = range(len(gains))
    ax.bar(positions, [gains[d] for d in sorted(gains)], width=0.62,
           color=CATEGORICAL[0], zorder=3)
    ax.set_xticks(list(positions))
    ax.set_xticklabels([str(d) for d in sorted(gains)])
    ax.axhline(0, color=GRID, linewidth=1)
    for position, dim in zip(positions, sorted(gains)):
        ax.annotate(f"{gains[dim]:.2e}", (position, gains[dim]), xytext=(0, 4),
                    textcoords="offset points", ha="center", color=INK, fontsize=8.5)
    plt.tight_layout()
    plt.show()

## 8. SDR-MRL training diagnostics

`sdr.diagnostics_every` recomputes, during training, the quantities the paper's
propositions are stated in — not just the loss. See the *Debugging SDR-MRL*
section of the repo README for what each one is for.

In [ ]:

import json
import math
from pathlib import Path

diagnostics_path = OUTPUT_DIR / "diagnostics.jsonl"

if not diagnostics_path.exists():
    print("No diagnostics.jsonl — only SDR-MRL writes one, and only when "
          "DIAGNOSTICS_EVERY > 0.")
    records = []
else:
    records = [json.loads(line) for line in diagnostics_path.read_text().splitlines()]
    last = records[-1]
    print(f"{len(records)} snapshots, last at step {last['step']}\n")
    print(f"teacher entropy      {last['teacher']['entropy']:.3f} nats "
          f"(perplexity {last['teacher']['perplexity']:.1f} of "
          f"{last['candidate_support']:.0f} candidates)")
    print(f"teacher mean cosine  {last['teacher']['mean_cosine']:+.3f}"
          "   — climbing toward 1 means the teacher is collapsing")
    print(f"D at full width      {last['full_dim_distortion']:.2e}"
          "   — must be ~0 when tau_T == tau_S")
    print(f"V_mono               {last['monotonicity_violation_rate']:.2f}")

In [ ]:
if records:
    prefixes = sorted(
        int(k.split("_")[1]) for k in records[0]["per_dim"]
        if int(k.split("_")[1]) < report["experiment"]["hidden_dim"]
    )
    # The ordinal ramp holds five distinguishable steps, so show the five lowest
    # rates — §7.3's regime of interest — and name what was left out.
    shown, omitted = prefixes[:5], prefixes[5:]
    steps = [record["step"] for record in records]

    figure, ax = new_axes(
        "Semantic distortion during training",
        "training step",
        "$\\mathcal{D}_k$  (in-batch, nats)",
        subtitle=("lightest = smallest prefix" +
                  (f" · omitted: {omitted}" if omitted else "")),
    )
    ends = []
    for index, dim in enumerate(shown):
        values = [record["per_dim"][f"dim_{dim}"]["distortion"] for record in records]
        ax.plot(steps, values, color=RATE_RAMP[index % len(RATE_RAMP)], linewidth=2, zorder=3)
        ends.append((values[-1], f"d={dim}"))
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax.set_xlim(steps[0], steps[-1] + max(1, (steps[-1] - steps[0])) * 0.14)
    label_line_ends(ax, steps[-1], ends)
    plt.tight_layout()
    plt.show()

    figure, ax = new_axes(
        "Teacher neighbourhood entropy",
        "training step",
        "$H(p_T)$  (nats)",
        subtitle="Eq 112 — near 0 the teacher is a hard-neighbour target; at the ceiling, uninformative",
    )
    ceiling = math.log(records[-1]["candidate_support"])
    ax.plot(steps, [record["teacher"]["entropy"] for record in records],
            color=CATEGORICAL[0], linewidth=2, zorder=3)
    ax.axhline(ceiling, color=INK_MUTED, linewidth=1, linestyle=(0, (4, 3)), zorder=2)
    ax.annotate(f"uniform ceiling  log({records[-1]['candidate_support']:.0f}) = {ceiling:.2f}",
                (steps[-1], ceiling), xytext=(0, 7), textcoords="offset points",
                color=INK_MUTED, fontsize=9, ha="right")
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax.set_ylim(0, ceiling * 1.18)
    plt.tight_layout()
    plt.show()

## 9. Check the implementation against the paper

35 assertions over independently constructed worlds: the gradient identities
(Eq 59/61/65), the variational decomposition on an exact discrete Markov chain
(Eq 30-32), successive refinement (Eq 39/42/45), unbiased rate sampling (Eq 68),
Theorem 1's linear-Gaussian optimum (Eq 75-84), and the rotation argument that
motivates the method (Eq 114-118).

In [ ]:

if VERIFY_MATH and Path("scripts/verify_sdr_math.py").exists():
    run([sys.executable, "scripts/verify_sdr_math.py"])
elif VERIFY_MATH:
    print("scripts/verify_sdr_math.py is not on this branch.")
else:
    print("skipped (VERIFY_MATH is off)")

## 10. Keep the results

In [ ]:

import shutil

if SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive/embedding-mrl") / RUN_NAME
    if destination.exists():
        shutil.rmtree(destination)
    shutil.copytree(OUTPUT_DIR, destination)
    print("copied to", destination)
else:
    archive = shutil.make_archive(f"/content/{RUN_NAME}", "zip", OUTPUT_DIR)
    print("archived to", archive)
    try:
        from google.colab import files

        files.download(archive)
    except ImportError:
        pass

print("\nrun directory contents:")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(OUTPUT_DIR)}  ({path.stat().st_size / 1024:.0f} KB)")